# Dance Genre Classification — Multi-Stream Feature Comparison
## AIST++ Dataset — 10 Genres

**Research question:** Does adding kinematic features (motion) improve classification across all 4 model families?

**Two experiments:**

| Experiment | Input | Features | Size |
|---|---|---|---|
| Baseline | Joint coords only | x, y per joint | 34 per frame |
| + Motion | Joint + Motion | x, y + velocity | 68 per frame |

**Motion** = how much each joint moved from the previous frame (dx, dy)

**Models:** SVM, Random Forest, LSTM, ST-GCN  
**Why:** SOTA methods (2s-AGCN, HD-GCN) use multi-stream inputs. We test whether this helps for dance genre classification on AIST++.

**Reference:** Shi et al. (2019) — Two-Stream Adaptive Graph Convolutional Networks (2s-AGCN)

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device.upper()}')


In [ ]:
!pip install scikit-learn matplotlib seaborn pandas numpy --quiet
print('All packages installed')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import os, pickle, time
import numpy as np
import pandas as pd
 
PKL_DIR = '/content/drive/MyDrive/AIST_PKL'
AR_DIR  = '/content/drive/MyDrive/AIST_ActionRec_MultiStream'
os.makedirs(AR_DIR, exist_ok=True)
 
GENRE_CODES = {
    'gBR': 'Break',    'gPO': 'Pop',        'gLO': 'Lock',
    'gWA': 'Waack',    'gMH': 'Mid Hip-hop', 'gLH': 'LA Hip-hop',
    'gHO': 'House',    'gKR': 'Krump',       'gJS': 'Street Jazz',
    'gJB': 'Ballet Jazz'
}
 
all_files = sorted(os.listdir(PKL_DIR))
print(f'Total .pkl files: {len(all_files)}')


In [ ]:
SEQUENCES = {}
for code, name in GENRE_CODES.items():
    sbm_files = sorted([f for f in all_files if f.startswith(code) and '_sBM_' in f])
    SEQUENCES[name] = {
        'code':  code,
        'train': sbm_files[:16],
        'test':  sbm_files[16:20]
    }
 
GENRES       = list(SEQUENCES.keys())
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
NUM_CLASSES  = len(GENRES)
 
print(f'Genres: {NUM_CLASSES}')
for name, s in SEQUENCES.items():
    print(f'  {name:15s}: {len(s["train"])} train | {len(s["test"])} test')


In [ ]:
COCO_EDGES = [
    (0,1),(0,2),(1,3),(2,4),(5,6),
    (5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]
NUM_JOINTS = 17
 
 
def normalize_skeleton(keypoints):
    """Normalize by hip center, scale to [-1,1]. Returns (17,2)."""
    xy = keypoints[:, :2].copy()
    if np.all(xy == 0) or np.any(np.isnan(xy)):
        return np.zeros((17, 2))
    hip_center = (xy[11] + xy[12]) / 2.0
    xy -= hip_center
    max_val = np.abs(xy).max()
    if max_val > 0:
        xy /= max_val
    return xy
 
 
def load_pkl_sequence(pkl_path, camera_idx=0, frame_step=6):
    """Load and normalize AIST++ .pkl file. Returns list of (17,2) arrays."""
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    kpts = data['keypoints2d'][camera_idx]
    return [normalize_skeleton(kpts[i]) for i in range(0, len(kpts), frame_step)]
 
 
def add_motion(frames):
    """
    Compute motion stream: displacement from previous frame.
    motion[t] = frames[t] - frames[t-1]
    motion[0] = zeros (no previous frame)

    Input:  list of (17,2) arrays, length T
    Output: list of (17,2) arrays, length T (motion only)
    """
    motion = [np.zeros((17, 2))]
    for t in range(1, len(frames)):
        motion.append(frames[t] - frames[t-1])
    return motion
 
 
def make_windows(frames, motion, window_size=55, stride=10):
    """
    Create sliding windows in 3 formats:
      flat_j:    (N, 55*34)       for SVM/RF — joint only
      flat_jm:   (N, 55*68)       for SVM/RF — joint + motion
      lstm_j:    (N, 55, 34)      for LSTM — joint only
      lstm_jm:   (N, 55, 68)      for LSTM — joint + motion
      gcn_j:     (N, 2, 55, 17)   for ST-GCN — joint only
      gcn_jm:    (N, 4, 55, 17)   for ST-GCN — joint + motion (4 channels)
    """
    flat_j, flat_jm = [], []
    lstm_j, lstm_jm = [], []
    gcn_j,  gcn_jm  = [], []
 
    T = len(frames)
    for s in range(0, T - window_size + 1, stride):
        j = np.array(frames[s:s+window_size])
        m = np.array(motion[s:s+window_size])
 
        
        jm = np.concatenate([j, m], axis=-1)
 
        
        flat_j.append(j.reshape(-1))
        flat_jm.append(jm.reshape(-1))
 
        
        lstm_j.append(j.reshape(window_size, -1))
        lstm_jm.append(jm.reshape(window_size, -1))
 
        
        gcn_j.append(j.transpose(2, 0, 1))
        gcn_jm.append(jm.transpose(2, 0, 1))
 
    def to_arr(lst, empty_shape):
        return np.array(lst) if lst else np.empty(empty_shape)
 
    return (
        to_arr(flat_j,  (0, window_size*34)),
        to_arr(flat_jm, (0, window_size*68)),
        to_arr(lstm_j,  (0, window_size, 34)),
        to_arr(lstm_jm, (0, window_size, 68)),
        to_arr(gcn_j,   (0, 2, window_size, 17)),
        to_arr(gcn_jm,  (0, 4, window_size, 17)),
    )
 
 
test_file   = f"{PKL_DIR}/{SEQUENCES['Break']['train'][0]}"
test_frames = load_pkl_sequence(test_file)
test_motion = add_motion(test_frames)
fj, fjm, lj, ljm, gj, gjm = make_windows(test_frames, test_motion)
 
print(f'Frames: {len(test_frames)}')
print(f'Joint flat:         {fj.shape}')
print(f'Joint+Motion flat:  {fjm.shape}')
print(f'Joint LSTM:         {lj.shape}')
print(f'Joint+Motion LSTM:  {ljm.shape}')
print(f'Joint GCN:          {gj.shape}')
print(f'Joint+Motion GCN:   {gjm.shape}')


In [ ]:
WINDOW_SIZE = 55
STRIDE      = 10
 
Xfj_tr, Xfjm_tr = [], []
Xlj_tr, Xljm_tr = [], []
Xgj_tr, Xgjm_tr = [], []
y_train = []
 
Xfj_te, Xfjm_te = [], []
Xlj_te, Xljm_te = [], []
Xgj_te, Xgjm_te = [], []
y_test = []
 
print('Loading 10 genres...')
print('=' * 65)
 
for genre, splits in SEQUENCES.items():
    label = GENRE_TO_IDX[genre]
    tr_win = te_win = 0
 
    for split, xf, xfm, xl, xlm, xg, xgm, y_list in [
        ('train', Xfj_tr, Xfjm_tr, Xlj_tr, Xljm_tr, Xgj_tr, Xgjm_tr, y_train),
        ('test',  Xfj_te, Xfjm_te, Xlj_te, Xljm_te, Xgj_te, Xgjm_te, y_test)
    ]:
        for pkl_file in splits[split]:
            path = f'{PKL_DIR}/{pkl_file}'
            if not os.path.exists(path):
                continue
            frames = load_pkl_sequence(path)
            motion = add_motion(frames)
            fj, fjm, lj, ljm, gj, gjm = make_windows(frames, motion, WINDOW_SIZE, STRIDE)
            if len(fj) == 0:
                continue
            xf.extend(fj);   xfm.extend(fjm)
            xl.extend(lj);   xlm.extend(ljm)
            xg.extend(gj);   xgm.extend(gjm)
            y_list.extend([label] * len(fj))
            if split == 'train': tr_win += len(fj)
            else: te_win += len(fj)
 
    print(f'{genre:15s}: train={tr_win:3d} | test={te_win:3d} windows')
 
Xfj_tr  = np.nan_to_num(np.array(Xfj_tr))
Xfjm_tr = np.nan_to_num(np.array(Xfjm_tr))
Xlj_tr  = np.nan_to_num(np.array(Xlj_tr))
Xljm_tr = np.nan_to_num(np.array(Xljm_tr))
Xgj_tr  = np.nan_to_num(np.array(Xgj_tr))
Xgjm_tr = np.nan_to_num(np.array(Xgjm_tr))
y_train = np.array(y_train)
 
Xfj_te  = np.nan_to_num(np.array(Xfj_te))
Xfjm_te = np.nan_to_num(np.array(Xfjm_te))
Xlj_te  = np.nan_to_num(np.array(Xlj_te))
Xljm_te = np.nan_to_num(np.array(Xljm_te))
Xgj_te  = np.nan_to_num(np.array(Xgj_te))
Xgjm_te = np.nan_to_num(np.array(Xgjm_te))
y_test  = np.array(y_test)
 
print('=' * 65)
print(f'Train: {len(y_train)} windows | Test: {len(y_test)} windows')
print(f'Joint flat:        {Xfj_tr.shape}')
print(f'Joint+Motion flat: {Xfjm_tr.shape}')


In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
 
results = {}
 
scaler_j = StandardScaler()
Xtr_j = np.nan_to_num(scaler_j.fit_transform(Xfj_tr))
Xte_j = np.nan_to_num(scaler_j.transform(Xfj_te))
 
t0  = time.time()
svm_j = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_j.fit(Xtr_j, y_train)
pred = svm_j.predict(Xte_j)
results['SVM_Joint'] = {
    'acc': accuracy_score(y_test, pred)*100,
    'f1':  f1_score(y_test, pred, average='macro')*100,
    'time': time.time()-t0
}
print(f'SVM Joint:        {results["SVM_Joint"]["acc"]:.2f}%')
 
scaler_jm = StandardScaler()
Xtr_jm = np.nan_to_num(scaler_jm.fit_transform(Xfjm_tr))
Xte_jm = np.nan_to_num(scaler_jm.transform(Xfjm_te))
 
t0  = time.time()
svm_jm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_jm.fit(Xtr_jm, y_train)
pred = svm_jm.predict(Xte_jm)
results['SVM_Joint+Motion'] = {
    'acc': accuracy_score(y_test, pred)*100,
    'f1':  f1_score(y_test, pred, average='macro')*100,
    'time': time.time()-t0
}
print(f'SVM Joint+Motion: {results["SVM_Joint+Motion"]["acc"]:.2f}%')
 
t0 = time.time()
rf_j = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_j.fit(Xfj_tr, y_train)
pred = rf_j.predict(Xfj_te)
results['RF_Joint'] = {
    'acc': accuracy_score(y_test, pred)*100,
    'f1':  f1_score(y_test, pred, average='macro')*100,
    'time': time.time()-t0
}
print(f'RF  Joint:        {results["RF_Joint"]["acc"]:.2f}%')
 
t0 = time.time()
rf_jm = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_jm.fit(Xfjm_tr, y_train)
pred = rf_jm.predict(Xfjm_te)
results['RF_Joint+Motion'] = {
    'acc': accuracy_score(y_test, pred)*100,
    'f1':  f1_score(y_test, pred, average='macro')*100,
    'time': time.time()-t0
}
print(f'RF  Joint+Motion: {results["RF_Joint+Motion"]["acc"]:.2f}%')


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
 
class SkeletonDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]
 
 
class DanceGenreLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=256, num_layers=2,
                 num_classes=10, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))
 
 
def train_eval(X_train, X_test, y_train, y_test, input_size,
               num_epochs=150, batch_size=64, label=''):
    model = DanceGenreLSTM(input_size=input_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = StepLR(optimizer, step_size=50, gamma=0.5)
 
    tr_loader = DataLoader(SkeletonDataset(X_train, y_train),
                           batch_size=batch_size, shuffle=True)
    te_loader = DataLoader(SkeletonDataset(X_test,  y_test),
                           batch_size=batch_size, shuffle=False)
 
    best_acc = 0.0
    history  = []
 
    for epoch in range(num_epochs):
        model.train()
        for X_b, y_b in tr_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
 
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X_b, y_b in te_loader:
                out = model(X_b.to(device))
                correct += (out.argmax(1) == y_b.to(device)).sum().item()
                total   += y_b.size(0)
        acc = 100 * correct / total
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f'best_lstm_{label}.pt')
        history.append(acc)
 
        if (epoch+1) % 30 == 0:
            print(f'  [{label}] Epoch {epoch+1}/{num_epochs} | Test: {acc:.1f}%')
 
    
    model.load_state_dict(torch.load(f'best_lstm_{label}.pt'))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in te_loader:
            out = model(X_b.to(device))
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(y_b.numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    final_acc  = accuracy_score(all_labels, all_preds) * 100
    final_f1   = f1_score(all_labels, all_preds, average='macro') * 100
 
    return final_acc, final_f1, history
 
 
print('Training LSTM — Joint only...')
lstm_j_acc, lstm_j_f1, lstm_j_hist = train_eval(
    Xlj_tr, Xlj_te, y_train, y_test, input_size=34, label='j'
)
results['LSTM_Joint'] = {'acc': lstm_j_acc, 'f1': lstm_j_f1}
print(f'LSTM Joint:        {lstm_j_acc:.2f}%')
 
print('\nTraining LSTM — Joint + Motion...')
lstm_jm_acc, lstm_jm_f1, lstm_jm_hist = train_eval(
    Xljm_tr, Xljm_te, y_train, y_test, input_size=68, label='jm'
)
results['LSTM_Joint+Motion'] = {'acc': lstm_jm_acc, 'f1': lstm_jm_f1}
print(f'LSTM Joint+Motion: {lstm_jm_acc:.2f}%')


In [ ]:
import torch.nn.functional as F
 
 
def build_adjacency(num_joints=17):
    A = np.zeros((num_joints, num_joints))
    for i in range(num_joints): A[i,i] = 1
    for i,j in COCO_EDGES:
        A[i,j] = 1; A[j,i] = 1
    D = np.diag(A.sum(axis=1) ** -0.5)
    return torch.FloatTensor(D @ A @ D)
 
 
class GraphConv(nn.Module):
    def __init__(self, in_ch, out_ch, A):
        super().__init__()
        self.A    = A
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1)
    def forward(self, x):
        A = self.A.to(x.device)
        return self.conv(torch.einsum('nctv,vw->nctw', x, A))
 
 
class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, stride=1, dropout=0.3):
        super().__init__()
        self.gcn = GraphConv(in_ch, out_ch, A)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, (9,1), padding=(4,0), stride=(stride,1)),
            nn.BatchNorm2d(out_ch), nn.Dropout(dropout)
        )
        self.residual = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=(stride,1)),
            nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch or stride != 1 else nn.Identity()
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.tcn(self.gcn(x)) + self.residual(x))
 
 
class STGCN(nn.Module):
    def __init__(self, in_channels=2, num_classes=10, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(in_channels * NUM_JOINTS)
        self.blocks  = nn.ModuleList([
            STGCNBlock(in_channels, 64,  A, dropout=dropout),
            STGCNBlock(64,  64,  A, dropout=dropout),
            STGCNBlock(64,  128, A, stride=2, dropout=dropout),
            STGCNBlock(128, 128, A, dropout=dropout),
            STGCNBlock(128, 256, A, stride=2, dropout=dropout),
            STGCNBlock(256, 256, A, dropout=dropout),
        ])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc   = nn.Linear(256, num_classes)
 
    def forward(self, x):
        N, C, T, V = x.shape
        x = x.permute(0,3,1,2).reshape(N, V*C, T)
        x = self.data_bn(x)
        x = x.reshape(N, V, C, T).permute(0, 2, 3, 1)
        for block in self.blocks: x = block(x)
        return self.fc(self.pool(x).reshape(N, -1))
 
 
def train_eval_gcn(X_train, X_test, y_train, y_test, in_channels,
                   num_epochs=150, batch_size=64, label=''):
    model = STGCN(in_channels=in_channels).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = StepLR(optimizer, step_size=50, gamma=0.5)
 
    tr_loader = DataLoader(SkeletonDataset(X_train, y_train),
                           batch_size=batch_size, shuffle=True)
    te_loader = DataLoader(SkeletonDataset(X_test,  y_test),
                           batch_size=batch_size, shuffle=False)
 
    best_acc = 0.0
    history  = []
 
    for epoch in range(num_epochs):
        model.train()
        for X_b, y_b in tr_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
 
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X_b, y_b in te_loader:
                out = model(X_b.to(device))
                correct += (out.argmax(1) == y_b.to(device)).sum().item()
                total   += y_b.size(0)
        acc = 100 * correct / total
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f'best_gcn_{label}.pt')
        history.append(acc)
 
        if (epoch+1) % 30 == 0:
            print(f'  [ST-GCN {label}] Epoch {epoch+1}/{num_epochs} | Test: {acc:.1f}%')
 
    model.load_state_dict(torch.load(f'best_gcn_{label}.pt'))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in te_loader:
            out = model(X_b.to(device))
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(y_b.numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    final_acc  = accuracy_score(all_labels, all_preds) * 100
    final_f1   = f1_score(all_labels, all_preds, average='macro') * 100
    return final_acc, final_f1, history
 
 
print('Training ST-GCN — Joint only (2 channels)...')
gcn_j_acc, gcn_j_f1, gcn_j_hist = train_eval_gcn(
    Xgj_tr, Xgj_te, y_train, y_test, in_channels=2, label='j'
)
results['STGCN_Joint'] = {'acc': gcn_j_acc, 'f1': gcn_j_f1}
print(f'ST-GCN Joint:        {gcn_j_acc:.2f}%')
 
print('\nTraining ST-GCN — Joint + Motion (4 channels)...')
gcn_jm_acc, gcn_jm_f1, gcn_jm_hist = train_eval_gcn(
    Xgjm_tr, Xgjm_te, y_train, y_test, in_channels=4, label='jm'
)
results['STGCN_Joint+Motion'] = {'acc': gcn_jm_acc, 'f1': gcn_jm_f1}
print(f'ST-GCN Joint+Motion: {gcn_jm_acc:.2f}%')


In [ ]:
import matplotlib.pyplot as plt
 
rows = []
for model in ['SVM', 'RF', 'LSTM', 'STGCN']:
    key_j  = f'{model}_Joint'
    key_jm = f'{model}_Joint+Motion'
    acc_j  = results[key_j]['acc']
    acc_jm = results[key_jm]['acc']
    f1_j   = results[key_j]['f1']
    f1_jm  = results[key_jm]['f1']
    rows.append({
        'Model':         'ST-GCN' if model == 'STGCN' else model,
        'Joint Acc':     round(acc_j, 2),
        '+Motion Acc':   round(acc_jm, 2),
        'Δ Acc':         round(acc_jm - acc_j, 2),
        'Joint F1':      round(f1_j, 2),
        '+Motion F1':    round(f1_jm, 2),
        'Δ F1':          round(f1_jm - f1_j, 2),
    })
 
df = pd.DataFrame(rows)
print('=' * 75)
print('MULTI-STREAM COMPARISON: Joint Only vs Joint + Motion')
print('=' * 75)
print(df.to_string(index=False))
df.to_csv('multistream_comparison.csv', index=False)
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_j  = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
colors_jm = ['#1565C0', '#2E7D32', '#E65100', '#AD1457']
model_names = ['SVM', 'RF', 'LSTM', 'ST-GCN']
x = range(4)
w = 0.35
 
accs_j  = [results[f'{m}_Joint']['acc']        for m in ['SVM','RF','LSTM','STGCN']]
accs_jm = [results[f'{m}_Joint+Motion']['acc'] for m in ['SVM','RF','LSTM','STGCN']]
bars1 = axes[0].bar([i-w/2 for i in x], accs_j,  w, label='Joint only',    color=colors_j, alpha=0.9)
bars2 = axes[0].bar([i+w/2 for i in x], accs_jm, w, label='Joint + Motion', color=colors_jm, alpha=0.9)
for b,v in zip(bars1, accs_j):  axes[0].text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontsize=8)
for b,v in zip(bars2, accs_jm): axes[0].text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontsize=8)
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(model_names)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy: Joint Only vs +Motion')
axes[0].legend(); axes[0].set_ylim(75, 105); axes[0].grid(axis='y', alpha=0.3)
 
deltas = [accs_jm[i] - accs_j[i] for i in range(4)]
bar_colors = ['#4CAF50' if d > 0 else '#F44336' for d in deltas]
bars = axes[1].bar(model_names, deltas, color=bar_colors, alpha=0.9)
for bar, v in zip(bars, deltas):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.05 if v>=0 else v-0.3,
                f'+{v:.2f}%' if v>=0 else f'{v:.2f}%', ha='center', fontweight='bold')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_ylabel('Accuracy change (%)')
axes[1].set_title('Improvement from Adding Motion')
axes[1].grid(axis='y', alpha=0.3)
 
axes[2].plot(lstm_j_hist,  color='#2196F3', label='LSTM Joint',       alpha=0.8)
axes[2].plot(lstm_jm_hist, color='#1565C0', label='LSTM +Motion',     alpha=0.8, linestyle='--')
axes[2].plot(gcn_j_hist,   color='#E91E63', label='ST-GCN Joint',     alpha=0.8)
axes[2].plot(gcn_jm_hist,  color='#AD1457', label='ST-GCN +Motion',   alpha=0.8, linestyle='--')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Test Accuracy (%)')
axes[2].set_title('Training Curves')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
 
plt.suptitle('Effect of Motion Features on Dance Genre Classification (AIST++)', fontsize=13)
plt.tight_layout()
plt.savefig('multistream_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved')


In [ ]:
import json, shutil
 
with open('multistream_results.json', 'w') as f:
    json.dump(results, f, indent=2)
 
files = [
    'multistream_comparison.csv',
    'multistream_results.png',
    'multistream_results.json',
    'best_lstm_j.pt', 'best_lstm_jm.pt',
    'best_gcn_j.pt',  'best_gcn_jm.pt'
]
for f in files:
    if os.path.exists(f):
        shutil.copy(f, f'{AR_DIR}/{f}')
        print(f'Saved: {f}')
 
print(f'\nAll results → AIST_ActionRec_MultiStream/')
print()
print('=' * 75)
print('FINAL SUMMARY')
print('=' * 75)
print(df.to_string(index=False))
print()
best_model = df.loc[df['+Motion Acc'].idxmax(), 'Model']
best_acc   = df['+Motion Acc'].max()
print(f'Best: {best_model} + Motion = {best_acc:.2f}%')
